In [9]:
# ──  Load Phase 1 Artifacts ──────────────────────────────────────────
import pickle
import numpy as np
import torch
from pathlib import Path

WORKING = Path("data/working")

def load(name: str):
    with open(WORKING / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

# ── Scalar constants ──────────────────────────────────────────────────────────
SEQUENCE_LENGTH = load("SEQUENCE_LENGTH")   # 15
MOVIE_FEAT_DIM  = load("MOVIE_FEAT_DIM")    # 792
NUM_USERS       = load("NUM_USERS")         # 6 040
NUM_MOVIES      = load("NUM_MOVIES")        # 3 706
BATCH_SIZE      = load("BATCH_SIZE")        # 256

# ── ID mappings ───────────────────────────────────────────────────────────────
user_id_to_idx  = load("user_id_to_idx")    # original UserID  → 0-based index
idx_to_user_id  = load("idx_to_user_id")    # 0-based index    → original UserID
movie_id_to_idx = load("movie_id_to_idx")   # original MovieID → 0-based index
idx_to_movie_id = load("idx_to_movie_id")   # 0-based index    → original MovieID

# ── Feature matrices ──────────────────────────────────────────────────────────
user_features   = load("user_features")     # shape (NUM_USERS,  3)
movie_features  = load("movie_features")    # shape (NUM_MOVIES, MOVIE_FEAT_DIM)
movies_enriched = load("movies_enriched")   # enriched DataFrame (for display / inference)
all_genres      = load("all_genres")        # list of 18 genre strings

# ── Split sequences ───────────────────────────────────────────────────────────
train_sequences = load("train_sequences")   # 636 443 samples
val_sequences   = load("val_sequences")     #  70 715 samples
test_sequences  = load("test_sequences")    # 202 451 samples

# ── User history (for inference / cold-start checks) ─────────────────────────
user_movie_history = load("user_movie_history")

# ── Raw train / test DataFrames (kept for reference) ─────────────────────────
train_data = load("train_data")
test_data  = load("test_data")

# ── Sanity check ─────────────────────────────────────────────────────────────
print("=" * 60)
print("PHASE 2 — ARTIFACTS LOADED")
print("=" * 60)
print(f"  NUM_USERS        : {NUM_USERS:,}")
print(f"  NUM_MOVIES       : {NUM_MOVIES:,}")
print(f"  MOVIE_FEAT_DIM   : {MOVIE_FEAT_DIM}")
print(f"  SEQUENCE_LENGTH  : {SEQUENCE_LENGTH}")
print(f"  BATCH_SIZE       : {BATCH_SIZE}")
print(f"  Train sequences  : {len(train_sequences):,}")
print(f"  Val   sequences  : {len(val_sequences):,}")
print(f"  Test  sequences  : {len(test_sequences):,}")
print(f"  user_features    : ({len(user_features)}, 3)  [users x (gender, age, occupation)]")
#print(f"  movie_features   : {np.array(movie_features).shape}")
feat_sample = movie_features['Feature_Vector'].iloc[0]
print(f"  movie_features   : ({len(movie_features)}, {len(feat_sample)})  [rows x feat_dim]")
print(f"  Device           : {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print("=" * 60)


PHASE 2 — ARTIFACTS LOADED
  NUM_USERS        : 6,040
  NUM_MOVIES       : 3,706
  MOVIE_FEAT_DIM   : 792
  SEQUENCE_LENGTH  : 15
  BATCH_SIZE       : 256
  Train sequences  : 636,443
  Val   sequences  : 70,715
  Test  sequences  : 202,451
  user_features    : (6040, 3)  [users x (gender, age, occupation)]
  movie_features   : (3884, 792)  [rows x feat_dim]
  Device           : cuda


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MovieSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.samples = []
        for s in sequences:
            uid  = int(s['user_id'])
            uidx = user_id_to_idx[uid]

            ctx_movie_idxs = [movie_id_to_idx[m] for m in s['seq_movie_ids']]
            ctx_ratings    = s['seq_ratings']
            ctx_positions  = s['seq_positions']
            tgt_movie_idx  = movie_id_to_idx[s['target_movie_id']]
            tgt_rating     = float(s['target_rating'])

            # Convert to tensors
            ctx_movie_idxs = torch.tensor(ctx_movie_idxs, dtype=torch.long)
            #ctx_ratings    = torch.tensor(ctx_ratings,    dtype=torch.float)
            # Keep as raw 1-5 ints for embedding lookup; padding zeros will use embedding 0
            ctx_ratings = torch.tensor([int(r) for r in ctx_ratings], dtype=torch.long)
            ctx_positions  = torch.tensor(ctx_positions,  dtype=torch.long)

            # Pad if sequence is shorter than SEQUENCE_LENGTH
            seq_len = len(ctx_movie_idxs)
            if seq_len < SEQUENCE_LENGTH:
                pad_len = SEQUENCE_LENGTH - seq_len
                ctx_movie_idxs = torch.cat([
                    ctx_movie_idxs,
                    torch.zeros(pad_len, dtype=torch.long)
                ])
                ctx_ratings = torch.cat([ctx_ratings, torch.zeros(pad_len, dtype=torch.long)])
                ctx_positions = torch.cat([
                    ctx_positions,
                    torch.zeros(pad_len, dtype=torch.long)
                ])

            self.samples.append({
                'user_idx'        : torch.tensor(uidx,         dtype=torch.long),
                'ctx_movie_idxs'  : ctx_movie_idxs,
                'ctx_ratings'     : ctx_ratings,
                'ctx_positions'   : ctx_positions,
                'target_movie_idx': torch.tensor(tgt_movie_idx, dtype=torch.long),
                'target_rating'   : torch.tensor(tgt_rating,    dtype=torch.float),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


train_dataset = MovieSequenceDataset(train_sequences)
val_dataset   = MovieSequenceDataset(val_sequences)
test_dataset  = MovieSequenceDataset(test_sequences)

# num_workers=0: all data is already pre-loaded into RAM as tensors,
# so multiprocessing workers add overhead and cause pin_memory thread crashes.
# pin_memory=False: same reason — no benefit when data is already in CPU RAM.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

# ── sanity check ─────────────────────────────────────────────────────────────
print(f"Train samples: {len(train_dataset):,}  |  batches: {len(train_loader):,}")
print(f"Val   samples: {len(val_dataset):,}  |  batches: {len(val_loader):,}")
print(f"Test  samples: {len(test_dataset):,}  |  batches: {len(test_loader):,}")

batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"  {k:20s} {v.shape}  dtype={v.dtype}")

print("\nDataLoaders ready (with padding to length 15).")


Train samples: 636,443  |  batches: 2,487
Val   samples: 70,715  |  batches: 277
Test  samples: 202,451  |  batches: 791
  user_idx             torch.Size([256])  dtype=torch.int64
  ctx_movie_idxs       torch.Size([256, 15])  dtype=torch.int64
  ctx_ratings          torch.Size([256, 15])  dtype=torch.int64
  ctx_positions        torch.Size([256, 15])  dtype=torch.int64
  target_movie_idx     torch.Size([256])  dtype=torch.int64
  target_rating        torch.Size([256])  dtype=torch.float32

DataLoaders ready (with padding to length 15).


In [8]:
# ── Model Architecture ────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ContentTower(nn.Module):
    """Content-based tower with learned embeddings for user demographics.
    
    Also accepts a collaborative_user_rep from the NCF rating encoder so the
    content tower can see WHAT the user has actually rated, not just WHO they are
    demographically. Demographics alone (gender/age/job) carry very little signal.
    """
    def __init__(self, movie_feat_dim=MOVIE_FEAT_DIM, hidden_dim=64):
        super().__init__()
        self.gender_emb     = nn.Embedding(3,  8)
        self.age_emb        = nn.Embedding(8,  16)
        self.occupation_emb = nn.Embedding(22, 32)
        # Demographic embedding dim: 8 + 16 + 32 = 56
        # Combined with collaborative_user_rep (hidden_dim): 56 + hidden_dim
        self.user_mlp = nn.Sequential(
            nn.Linear(56 + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.movie_mlp = nn.Sequential(
            nn.LayerNorm(movie_feat_dim),
            nn.Linear(movie_feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )

    def forward(self, user_features_raw, movie_features, collaborative_user_rep):
        """
        user_features_raw:      [B, 3]         — (gender, age, occupation) indices
        movie_features:         [B, movie_feat_dim]
        collaborative_user_rep: [B, hidden_dim] — user rep from NCF rating encoder
        returns: [B] content scores
        """
        gender_idx     = user_features_raw[:, 0].long()
        age_idx        = user_features_raw[:, 1].long()
        occupation_idx = user_features_raw[:, 2].long()

        demo_emb = torch.cat([
            self.gender_emb(gender_idx),         # [B, 8]
            self.age_emb(age_idx),               # [B, 16]
            self.occupation_emb(occupation_idx)  # [B, 32]
        ], dim=1)                                # [B, 56]

        # Concatenate demographic embedding with what the user has actually rated
        user_input = torch.cat([demo_emb, collaborative_user_rep], dim=1)  # [B, 56+hidden_dim]
        user_rep   = self.user_mlp(user_input)   # [B, hidden_dim]
        movie_rep  = self.movie_mlp(movie_features)  # [B, hidden_dim]

        score = (user_rep * movie_rep).sum(dim=1)    # [B]
        return score


   

class RatingEncoder(nn.Module):
    def __init__(self, num_movies, num_ratings=6, emb_dim=64, num_heads=4, shared_movie_emb=None):
        super().__init__()
        self.movie_emb  = shared_movie_emb or nn.Embedding(num_movies, emb_dim)
        self.rating_emb = nn.Embedding(num_ratings, emb_dim, padding_idx=0)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, emb_dim))  # learnable CLS
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.attention   = nn.MultiheadAttention(emb_dim, num_heads, batch_first=True, dropout=0.1)
        self.layer_norm1 = nn.LayerNorm(emb_dim)
        self.layer_norm2 = nn.LayerNorm(emb_dim)
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 2), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(emb_dim * 2, emb_dim),
        )
        self.dropout = nn.Dropout(0.2)

    def forward(self, movie_ids, ratings):
        B = movie_ids.shape[0]
        m_emb    = self.movie_emb(movie_ids)       # [B, 15, D]
        r_emb    = self.rating_emb(ratings)        # [B, 15, D]
        combined = self.dropout(m_emb + r_emb)    # [B, 15, D]
        
        cls = self.cls_token.expand(B, -1, -1)    # [B, 1, D]
        combined = torch.cat([cls, combined], dim=1)  # [B, 16, D]
        
        attn_out, _ = self.attention(combined, combined, combined)
        combined = self.layer_norm1(attn_out + combined)
        combined = self.layer_norm2(self.ffn(combined) + combined)
        
        return combined[:, 0, :]   # return CLS token output [B, D]



class NCFTower(nn.Module):
    def __init__(self, num_movies, emb_dim=64, shared_movie_emb=None):
        super().__init__()
        self.rating_encoder   = RatingEncoder(num_movies, emb_dim=emb_dim, shared_movie_emb=shared_movie_emb)
        self.target_movie_emb = shared_movie_emb if shared_movie_emb is not None else nn.Embedding(num_movies, emb_dim)
        self.gmf_linear = nn.Linear(emb_dim, 32)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.fusion = nn.Linear(64, 1)

    def score_from_user_rep(self, user_rep, target_movie_id):
            """Compute NCF score given a pre-computed user_rep. Used by HybridRecommender
            so the rating encoder runs only once and its output is shared with ContentTower."""
            movie_rep = self.target_movie_emb(target_movie_id)
            gmf_out   = self.gmf_linear(user_rep * movie_rep)
            mlp_out   = self.mlp(torch.cat([user_rep, movie_rep], dim=1))
            return self.fusion(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)

    def forward(self, ctx_movie_ids, ctx_ratings, target_movie_id):
        user_rep = self.rating_encoder(ctx_movie_ids, ctx_ratings)
        return self.score_from_user_rep(user_rep, target_movie_id)

        

class SequentialTower(nn.Module):
    def __init__(self, num_movies, seq_len=15, emb_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        # Own dedicated embeddings — NOT shared with NCF tower.
        # Sharing forces one embedding to simultaneously represent
        # collaborative identity (NCF) and temporal order (Sequential),
        # which pulls the gradients in conflicting directions.
        self.movie_emb  = nn.Embedding(num_movies, emb_dim, padding_idx=0)
        self.rating_emb = nn.Embedding(6, emb_dim, padding_idx=0)
        self.pos_emb    = nn.Embedding(seq_len, emb_dim)
        self.emb_dropout = nn.Dropout(0.3)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,   # scales automatically with emb_dim
            dropout=0.3,
            batch_first=True
        )
        self.transformer    = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attention_pool = nn.MultiheadAttention(emb_dim, num_heads, batch_first=True, dropout=0.2)
        self.out_dropout    = nn.Dropout(0.3)
        self.score_head     = nn.Linear(emb_dim, 1)


    
    def forward(self, movie_ids, ratings, positions):
            seq_input = self.emb_dropout(
                self.movie_emb(movie_ids) +
                self.rating_emb(ratings)  +
                self.pos_emb(positions)
            )                                              # [B, 15, D]
    
            # PyTorch 2.6 requires an explicit float attn_mask whenever is_causal
            # is involved — the SDPA fast path that avoids this is not reliably
            # reached with TransformerEncoder. generate_square_subsequent_mask
            # produces the correct upper-triangular float('-inf') mask directly
            # and works identically in both train and eval mode.
            seq_len = movie_ids.shape[1]
            causal_mask = nn.Transformer.generate_square_subsequent_mask(
                seq_len, device=movie_ids.device, dtype=seq_input.dtype
            )                                              # [15, 15], float, -inf above diagonal
    
            seq_out = self.transformer(
                seq_input,
                mask=causal_mask,
                src_key_padding_mask=None,
                is_causal=False
            )                                              # [B, 15, D]
    
            query  = seq_out.mean(dim=1, keepdim=True)                 # [B, 1, D]
            pooled, _ = self.attention_pool(query, seq_out, seq_out)   # [B, 1, D]
    
            score = self.score_head(self.out_dropout(pooled.squeeze(1))).squeeze(1)  # [B]
            return score
        





class HybridRecommender(nn.Module):
    def __init__(self, num_movies, movie_feat_dim, emb_dim=64):
        super().__init__()
        # Shared movie embedding used by both NCF and Sequential towers
        self.movie_emb = nn.Embedding(num_movies, emb_dim, padding_idx=0)

        self.content_tower = ContentTower(movie_feat_dim, hidden_dim=emb_dim)
        self.ncf_tower     = NCFTower(num_movies, emb_dim=emb_dim, shared_movie_emb=self.movie_emb)
        self.seq_tower     = SequentialTower(num_movies, seq_len=15, emb_dim=emb_dim,
                                                     num_heads=4, num_layers=2)
                #  seq_tower now has its own movie embedding — not shared with NCF.
        self.fusion_weights = nn.Parameter(torch.tensor([0.4, 0.4, 0.2]))
        
        self.output_head = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        # Initialize output bias to predict the mean rating (3.5 on 1-5 scale)
        # This dramatically reduces epoch 1 loss spike
        nn.init.constant_(self.output_head[2].bias, 3.5)
        nn.init.xavier_uniform_(self.output_head[0].weight)
        nn.init.xavier_uniform_(self.output_head[2].weight)
    
    def forward(self, user_features, ctx_movie_ids, ctx_ratings, ctx_positions,
                target_movie_id, target_movie_features):
        # Compute the collaborative user representation ONCE from the rating encoder.
        # It is then reused by both ContentTower and NCFTower, avoiding double computation.
        collab_user_rep = self.ncf_tower.rating_encoder(ctx_movie_ids, ctx_ratings)  # [B, D]

        content_score = self.content_tower(user_features, target_movie_features, collab_user_rep)
        ncf_score     = self.ncf_tower.score_from_user_rep(collab_user_rep, target_movie_id)
        seq_score     = self.seq_tower(ctx_movie_ids, ctx_ratings, ctx_positions)

        # Stack raw scores — no batch normalization here.
        # Batch norm makes predictions depend on other samples in the batch,
        # causing train/eval discrepancy. The output_head MLP learns centering itself.
        scores      = torch.stack([content_score, ncf_score, seq_score], dim=1)  # [B, 3]
        final_score = self.output_head(scores).squeeze(1)

        if not self.training:
            final_score = final_score.clamp(1.0, 5.0)
        return final_score, content_score, ncf_score, seq_score

print(" Model architecture defined")


 Model architecture defined


In [9]:
# ── Instantiate Model ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HybridRecommender(
    num_movies=NUM_MOVIES,
    movie_feat_dim=MOVIE_FEAT_DIM,
    emb_dim=128        # was 64; larger embeddings capture richer user/movie identity
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: HybridRecommender")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device: {device}")



Model: HybridRecommender
  Total parameters: 1,849,118
  Trainable parameters: 1,849,118
  Device: cuda


In [10]:
import os, pathlib
pathlib.Path('data/models').mkdir(parents=True, exist_ok=True)
for stale in ['data/models/best_model_final.pth', 'data/working/best_model_final.pth']:
    if os.path.exists(stale):
        os.remove(stale)
        print(f"Deleted: {stale}")
print("Clean. Ready to train.")

Clean. Ready to train.


In [11]:
# Remove the warmup lambda and LambdaLR from the training loop cell.
# Instead, build a single combined scheduler using SequentialLR:

from torch.optim.lr_scheduler import LinearLR, ReduceLROnPlateau, SequentialLR
criterion = nn.HuberLoss(delta=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
# Slightly lower LR for larger model — 5e-4 can cause instability at emb_dim=128
warmup  = LinearLR(optimizer, start_factor=0.33, end_factor=1.0, total_iters=3)

plateau = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-5)
# patience=2: reduce LR after 2 stagnant epochs instead of 3 — faster response

In [12]:
# ──  Training Loop ─────────────────────────────────────────────────
import time, math
import numpy as np
from tqdm.auto import tqdm

# Stack user demographic columns into a tensor for fast index lookup
user_features_array  = user_features[['Gender_Encoded', 'Age_Encoded', 'Occupation_Encoded']].values.astype(np.int64)
user_features_tensor = torch.tensor(user_features_array, dtype=torch.long)

# Stack movie feature vectors into a tensor for fast index lookup
# Align movie features to movie_id_to_idx order
aligned_features = np.zeros((NUM_MOVIES, MOVIE_FEAT_DIM), dtype=np.float32)
movie_feat_lookup = dict(zip(movie_features['MovieID'], movie_features['Feature_Vector']))
for movie_id, idx in movie_id_to_idx.items():
    if movie_id in movie_feat_lookup:
        aligned_features[idx] = movie_feat_lookup[movie_id].astype(np.float32)


zero_rows = (aligned_features.sum(axis=1) == 0).sum()
print(f"Movies with zero feature vector: {zero_rows}")
# This should be near 0 — if not, some rated movies have no feature vector

movie_features_tensor = torch.tensor(aligned_features, dtype=torch.float32)
print(f"movie_features_tensor shape: {movie_features_tensor.shape}")  # should be (3706, 792)

print(f"user_features_tensor  shape: {user_features_tensor.shape}")
print(f"movie_features_tensor shape: {movie_features_tensor.shape}")


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training", leave=False):
        user_idx         = batch['user_idx'].to(device)
        ctx_movie_idxs   = batch['ctx_movie_idxs'].to(device)
        
        ctx_ratings      = batch['ctx_ratings'].to(device)

            
        ctx_positions    = batch['ctx_positions'].to(device)
        target_movie_idx = batch['target_movie_idx'].to(device)
        target_rating    = batch['target_rating'].to(device)

        uf  = user_features_tensor[user_idx.cpu()].to(device)
        tmf = movie_features_tensor[target_movie_idx.cpu()].to(device)

        optimizer.zero_grad()
        pred, _, _, _ = model(uf, ctx_movie_idxs, ctx_ratings,
                              ctx_positions, target_movie_idx, tmf)
        loss = criterion(pred, target_rating)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Fix 8: gradient clipping
        optimizer.step()

        total_loss += loss.item() * len(target_rating)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_huber_loss = 0
    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            user_idx         = batch['user_idx'].to(device)
            ctx_movie_idxs   = batch['ctx_movie_idxs'].to(device)
            ctx_ratings      = batch['ctx_ratings'].to(device)
            ctx_positions    = batch['ctx_positions'].to(device)
            target_movie_idx = batch['target_movie_idx'].to(device)
            target_rating    = batch['target_rating'].to(device)

            uf  = user_features_tensor[user_idx.cpu()].to(device)
            tmf = movie_features_tensor[target_movie_idx.cpu()].to(device)

            pred, _, _, _ = model(uf, ctx_movie_idxs, ctx_ratings,
                                  ctx_positions, target_movie_idx, tmf)
            loss = criterion(pred, target_rating)

            total_huber_loss += loss.item() * len(target_rating)
            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target_rating.cpu().numpy())

    avg_huber      = total_huber_loss / len(loader.dataset)
    all_preds_np   = np.array(all_preds)
    all_targets_np = np.array(all_targets)
    mse            = np.mean((all_preds_np - all_targets_np) ** 2)
    rmse           = math.sqrt(mse)
    mae            = np.abs(all_preds_np - all_targets_np).mean()

    return avg_huber, rmse, mae

    

# ── Training configuration ──────────────────────────────────────────────────
NUM_EPOCHS          = 30
EARLY_STOP_PATIENCE = 7
best_val_loss       = float('inf')
patience_counter    = 0
history             = []

print(f"Starting training for {NUM_EPOCHS} epochs...")
print(f"Early stopping patience: {EARLY_STOP_PATIENCE}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_loss                  = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_rmse, val_mae = evaluate(model, val_loader, criterion, device)

    # Fix 6: clean warmup -> plateau handoff using schedulers defined in previous cell
    if epoch <= 3:
        warmup.step()
    else:
        plateau.step(val_loss)

    epoch_time = time.time() - epoch_start

    history.append({
        'epoch':      epoch,
        'train_loss': train_loss,
        'val_loss':   val_loss,
        'val_rmse':   val_rmse,
        'val_mae':    val_mae,
        'lr':         optimizer.param_groups[0]['lr']
    })

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val RMSE: {val_rmse:.4f} | "
          f"Val MAE: {val_mae:.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
          f"Time: {epoch_time:.1f}s")
    
    oh = model.output_head
    w1 = oh[0].weight.data.cpu()   # shape [16, 3]
    # Column importance: how much each tower feeds into the hidden layer
    tower_importance = w1.abs().mean(dim=0)  # [3]
    tower_importance = tower_importance / tower_importance.sum()
    print(f"           Tower importance (output_head): content={tower_importance[0]:.3f}  ncf={tower_importance[1]:.3f}  seq={tower_importance[2]:.3f}")

    
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        # Fix 9: save scheduler states so training can be resumed cleanly
        torch.save({
            'epoch':                   epoch,
            'model_state_dict':        model.state_dict(),
            'optimizer_state_dict':    optimizer.state_dict(),
            'warmup_scheduler_state':  warmup.state_dict(),
            'plateau_scheduler_state': plateau.state_dict(),
            'val_loss':                val_loss,
            'val_rmse':                val_rmse,
            'history':                 history,
            'fusion_weights':          model.fusion_weights.data.clone()
        }, 'data/models/best_model_final.pth')
        print(f"  -> Best model saved (val_loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs")
            break

print("\nTraining complete")

Movies with zero feature vector: 0
movie_features_tensor shape: torch.Size([3706, 792])
user_features_tensor  shape: torch.Size([6040, 3])
movie_features_tensor shape: torch.Size([3706, 792])
Starting training for 30 epochs...
Early stopping patience: 7



Training:   0%|          | 0/2487 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/277 [00:00<?, ?it/s]

Epoch  1/30 | Train Loss: 0.2723 | Val Loss: 0.2545 | Val RMSE: 0.9344 | Val MAE: 0.7173 | LR: 0.000166 | Time: 100.3s
           Tower importance (output_head): content=0.297  ncf=0.299  seq=0.404
  -> Best model saved (val_loss: 0.2545)


Training:   0%|          | 0/2487 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/277 [00:00<?, ?it/s]

Epoch 18/30 | Train Loss: 0.2334 | Val Loss: 0.2398 | Val RMSE: 0.8928 | Val MAE: 0.6865 | LR: 0.000150 | Time: 98.2s
           Tower importance (output_head): content=0.273  ncf=0.290  seq=0.436


Training:   0%|          | 0/2487 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/277 [00:00<?, ?it/s]

Epoch 19/30 | Train Loss: 0.2322 | Val Loss: 0.2401 | Val RMSE: 0.8930 | Val MAE: 0.6871 | LR: 0.000150 | Time: 98.9s
           Tower importance (output_head): content=0.272  ncf=0.292  seq=0.436

Early stopping triggered after 19 epochs

Training complete


In [13]:
# ── Test Set Evaluation ──────────────────────────────────────────────────────
print("Loading best model checkpoint...")
checkpoint = torch.load('data/models/best_model_final.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Best checkpoint: epoch {checkpoint['epoch']}, val_rmse={checkpoint['val_rmse']:.4f}")

test_loss, test_rmse, test_mae = evaluate(model, test_loader, criterion, device)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"  Test Loss : {test_loss:.4f}")
print(f"  Test RMSE : {test_rmse:.4f}   (target: 0.85–0.95)")
print(f"  Test MAE  : {test_mae:.4f}")

# Fusion weight analysis

oh = model.output_head
w1 = oh[0].weight.data.cpu()
tower_importance = w1.abs().mean(dim=0)
tower_importance = tower_importance / tower_importance.sum()
print(f"\n  Tower importance (learned by output_head):")
print(f"    Content tower : {tower_importance[0]:.3f}  ({tower_importance[0]*100:.1f}%)")
print(f"    NCF tower     : {tower_importance[1]:.3f}  ({tower_importance[1]*100:.1f}%)")
print(f"    Sequential    : {tower_importance[2]:.3f}  ({tower_importance[2]*100:.1f}%)")

Loading best model checkpoint...
Best checkpoint: epoch 12, val_rmse=0.8837


Evaluating:   0%|          | 0/791 [00:00<?, ?it/s]


TEST SET RESULTS
  Test Loss : 0.2420
  Test RMSE : 0.8954   (target: 0.85–0.95)
  Test MAE  : 0.6920

  Tower importance (learned by output_head):
    Content tower : 0.288  (28.8%)
    NCF tower     : 0.278  (27.8%)
    Sequential    : 0.433  (43.3%)


In [10]:
# ── Session Restore ───────────────────────────────────────────────
# Run this after a kernel restart to reload all state without retraining.
# Requires: data/working/*.pkl  and  data/models/best_model_final.pth
import pickle, math, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

WORKING = Path("data/working")

def load(name):
    with open(WORKING / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

# ── Constants & mappings ──────────────────────────────────────────
SEQUENCE_LENGTH  = load("SEQUENCE_LENGTH")
MOVIE_FEAT_DIM   = load("MOVIE_FEAT_DIM")
NUM_USERS        = load("NUM_USERS")
NUM_MOVIES       = load("NUM_MOVIES")
BATCH_SIZE       = load("BATCH_SIZE")
user_id_to_idx   = load("user_id_to_idx")
idx_to_user_id   = load("idx_to_user_id")
movie_id_to_idx  = load("movie_id_to_idx")
idx_to_movie_id  = load("idx_to_movie_id")
user_features    = load("user_features")
movie_features   = load("movie_features")
movies_enriched  = load("movies_enriched")
all_genres       = load("all_genres")
train_sequences  = load("train_sequences")
val_sequences    = load("val_sequences")
test_sequences   = load("test_sequences")
user_movie_history = load("user_movie_history")
train_data       = load("train_data")
test_data        = load("test_data")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Artifacts loaded. Device: {device}")

# ── Rebuild feature tensors ───────────────────────────────────────
user_features_array  = user_features[['Gender_Encoded','Age_Encoded','Occupation_Encoded']].values.astype(np.int64)
user_features_tensor = torch.tensor(user_features_array, dtype=torch.long)

aligned_features  = np.zeros((NUM_MOVIES, MOVIE_FEAT_DIM), dtype=np.float32)
movie_feat_lookup = dict(zip(movie_features['MovieID'], movie_features['Feature_Vector']))
for movie_id, idx in movie_id_to_idx.items():
    if movie_id in movie_feat_lookup:
        aligned_features[idx] = movie_feat_lookup[movie_id].astype(np.float32)
movie_features_tensor = torch.tensor(aligned_features, dtype=torch.float32)
print(f"user_features_tensor : {user_features_tensor.shape}")
print(f"movie_features_tensor: {movie_features_tensor.shape}")

# ── Rebuild model architecture (must match training exactly) ──────
class ContentTower(nn.Module):
    def __init__(self, movie_feat_dim, hidden_dim):
        super().__init__()
        self.gender_emb     = nn.Embedding(3,  8)
        self.age_emb        = nn.Embedding(8,  16)
        self.occupation_emb = nn.Embedding(22, 32)
        self.user_mlp = nn.Sequential(
            nn.Linear(56 + hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.movie_mlp = nn.Sequential(
            nn.LayerNorm(movie_feat_dim), nn.Linear(movie_feat_dim, 256),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
    def forward(self, user_features_raw, movie_features, collaborative_user_rep):
        g = self.gender_emb(user_features_raw[:, 0].long())
        a = self.age_emb(user_features_raw[:, 1].long())
        o = self.occupation_emb(user_features_raw[:, 2].long())
        demo_emb   = torch.cat([g, a, o], dim=1)
        user_input = torch.cat([demo_emb, collaborative_user_rep], dim=1)
        user_rep   = self.user_mlp(user_input)
        movie_rep  = self.movie_mlp(movie_features)
        return (user_rep * movie_rep).sum(dim=1)

class RatingEncoder(nn.Module):
    def __init__(self, num_movies, num_ratings=6, emb_dim=64, num_heads=4, shared_movie_emb=None):
        super().__init__()
        self.movie_emb   = shared_movie_emb or nn.Embedding(num_movies, emb_dim)
        self.rating_emb  = nn.Embedding(num_ratings, emb_dim, padding_idx=0)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, emb_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.attention   = nn.MultiheadAttention(emb_dim, num_heads, batch_first=True, dropout=0.1)
        self.layer_norm1 = nn.LayerNorm(emb_dim)
        self.layer_norm2 = nn.LayerNorm(emb_dim)
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 2), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(emb_dim * 2, emb_dim)
        )
        self.dropout = nn.Dropout(0.2)
    def forward(self, movie_ids, ratings):
        B        = movie_ids.shape[0]
        combined = self.dropout(self.movie_emb(movie_ids) + self.rating_emb(ratings))
        cls      = self.cls_token.expand(B, -1, -1)
        combined = torch.cat([cls, combined], dim=1)
        attn_out, _ = self.attention(combined, combined, combined)
        combined = self.layer_norm1(attn_out + combined)
        combined = self.layer_norm2(self.ffn(combined) + combined)
        return combined[:, 0, :]

class NCFTower(nn.Module):
    def __init__(self, num_movies, emb_dim=64, shared_movie_emb=None):
        super().__init__()
        self.rating_encoder   = RatingEncoder(num_movies, emb_dim=emb_dim, shared_movie_emb=shared_movie_emb)
        self.target_movie_emb = shared_movie_emb or nn.Embedding(num_movies, emb_dim)
        self.gmf_linear = nn.Linear(emb_dim, 32)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2)
        )
        self.fusion = nn.Linear(64, 1)
    def score_from_user_rep(self, user_rep, target_movie_id):
        movie_rep = self.target_movie_emb(target_movie_id)
        gmf_out   = self.gmf_linear(user_rep * movie_rep)
        mlp_out   = self.mlp(torch.cat([user_rep, movie_rep], dim=1))
        return self.fusion(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)
    def forward(self, ctx_movie_ids, ctx_ratings, target_movie_id):
        return self.score_from_user_rep(
            self.rating_encoder(ctx_movie_ids, ctx_ratings), target_movie_id)

class SequentialTower(nn.Module):
    def __init__(self, num_movies, seq_len=15, emb_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        self.movie_emb   = nn.Embedding(num_movies, emb_dim, padding_idx=0)
        self.rating_emb  = nn.Embedding(6, emb_dim, padding_idx=0)
        self.pos_emb     = nn.Embedding(seq_len, emb_dim)
        self.emb_dropout = nn.Dropout(0.3)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=num_heads,
            dim_feedforward=emb_dim * 4, dropout=0.3, batch_first=True
        )
        self.transformer    = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attention_pool = nn.MultiheadAttention(emb_dim, num_heads, batch_first=True, dropout=0.2)
        self.out_dropout    = nn.Dropout(0.3)
        self.score_head     = nn.Linear(emb_dim, 1)
    def forward(self, movie_ids, ratings, positions):
        seq_input   = self.emb_dropout(
            self.movie_emb(movie_ids) + self.rating_emb(ratings) + self.pos_emb(positions)
        )
        seq_len     = movie_ids.shape[1]
        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            seq_len, device=movie_ids.device, dtype=seq_input.dtype
        )
        seq_out     = self.transformer(seq_input, mask=causal_mask,
                                       src_key_padding_mask=None, is_causal=False)
        query       = seq_out.mean(dim=1, keepdim=True)
        pooled, _   = self.attention_pool(query, seq_out, seq_out)
        return self.score_head(self.out_dropout(pooled.squeeze(1))).squeeze(1)

class HybridRecommender(nn.Module):
    def __init__(self, num_movies, movie_feat_dim, emb_dim=64):
        super().__init__()
        self.movie_emb      = nn.Embedding(num_movies, emb_dim, padding_idx=0)
        self.content_tower  = ContentTower(movie_feat_dim, hidden_dim=emb_dim)
        self.ncf_tower      = NCFTower(num_movies, emb_dim=emb_dim, shared_movie_emb=self.movie_emb)
        self.seq_tower      = SequentialTower(num_movies, seq_len=15, emb_dim=emb_dim, num_heads=4, num_layers=2)
        self.fusion_weights = nn.Parameter(torch.tensor([0.4, 0.4, 0.2]))
        self.output_head    = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))
        nn.init.constant_(self.output_head[2].bias, 3.5)
        nn.init.xavier_uniform_(self.output_head[0].weight)
        nn.init.xavier_uniform_(self.output_head[2].weight)
    def forward(self, user_features, ctx_movie_ids, ctx_ratings, ctx_positions,
                target_movie_id, target_movie_features):
        collab_user_rep = self.ncf_tower.rating_encoder(ctx_movie_ids, ctx_ratings)
        content_score   = self.content_tower(user_features, target_movie_features, collab_user_rep)
        ncf_score       = self.ncf_tower.score_from_user_rep(collab_user_rep, target_movie_id)
        seq_score       = self.seq_tower(ctx_movie_ids, ctx_ratings, ctx_positions)
        scores          = torch.stack([content_score, ncf_score, seq_score], dim=1)
        final_score     = self.output_head(scores).squeeze(1)
        if not self.training:
            final_score = final_score.clamp(1.0, 5.0)
        return final_score, content_score, ncf_score, seq_score

# ── Load checkpoint into model ────────────────────────────────────
model = HybridRecommender(
    num_movies=NUM_MOVIES, movie_feat_dim=MOVIE_FEAT_DIM, emb_dim=128
).to(device)

checkpoint = torch.load('data/models/best_model_final.pth',
                        map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Checkpoint loaded — epoch {checkpoint['epoch']}, val_rmse={checkpoint['val_rmse']:.4f}")

# ── Rebuild DataLoaders ───────────────────────────────────────────
class MovieSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.samples = []
        for s in sequences:
            uidx          = user_id_to_idx[int(s['user_id'])]
            ctx_movie_idxs = torch.tensor([movie_id_to_idx[m] for m in s['seq_movie_ids']], dtype=torch.long)
            ctx_ratings    = torch.tensor([int(r) for r in s['seq_ratings']],              dtype=torch.long)
            ctx_positions  = torch.tensor(s['seq_positions'],                               dtype=torch.long)
            tgt_movie_idx  = movie_id_to_idx[s['target_movie_id']]
            self.samples.append({
                'user_idx'        : torch.tensor(uidx,          dtype=torch.long),
                'ctx_movie_idxs'  : ctx_movie_idxs,
                'ctx_ratings'     : ctx_ratings,
                'ctx_positions'   : ctx_positions,
                'target_movie_idx': torch.tensor(tgt_movie_idx, dtype=torch.long),
                'target_rating'   : torch.tensor(float(s['target_rating']), dtype=torch.float),
            })
    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

train_dataset = MovieSequenceDataset(train_sequences)
val_dataset   = MovieSequenceDataset(val_sequences)
test_dataset  = MovieSequenceDataset(test_sequences)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f"DataLoaders ready — test batches: {len(test_loader):,}")

# ── Criterion (needed by evaluate() in cells 8-10) ───────────────
criterion = nn.HuberLoss(delta=0.5)

print("\nSession fully restored.")

Artifacts loaded. Device: cuda
user_features_tensor : torch.Size([6040, 3])
movie_features_tensor: torch.Size([3706, 792])
Checkpoint loaded — epoch 12, val_rmse=0.8837
DataLoaders ready — test batches: 791

Session fully restored.


In [3]:
# ──Training History Plot ─────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = checkpoint['history']

epochs     = [h['epoch']      for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss']   for h in history]
val_rmse   = [h['val_rmse']   for h in history]
lrs        = [h['lr']         for h in history]
best_epoch = checkpoint['epoch']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — Hybrid Movie Recommender', fontsize=14, fontweight='bold')

axes[0].plot(epochs, train_loss, label='Train Huber Loss', color='steelblue',  linewidth=2)
axes[0].plot(epochs, val_loss,   label='Val Huber Loss',   color='darkorange', linewidth=2)
axes[0].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.6, label=f'Best (epoch {best_epoch})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Huber Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_rmse, color='green', linewidth=2, marker='o', markersize=3)
axes[1].axhline(y=0.95, color='red',   linestyle='--', alpha=0.6, label='Target ceiling (0.95)')
axes[1].axhline(y=0.85, color='green', linestyle='--', alpha=0.6, label='Target floor   (0.85)')
axes[1].axvline(x=best_epoch, color='red', linestyle=':', alpha=0.6)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('RMSE')
axes[1].set_title('Validation RMSE'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, lrs, color='purple', linewidth=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule'); axes[2].grid(alpha=0.3)
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig('data/models/training_history.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: data/models/training_history.png")

print("\n{:<8} {:>12} {:>12} {:>10} {:>10}".format("Epoch","Train Loss","Val Loss","Val RMSE","Val MAE"))
print("-" * 56)
for h in history:
    marker = " <-- best" if h["epoch"] == best_epoch else ""
    print("{:<8} {:>12.4f} {:>12.4f} {:>10.4f} {:>10.4f}{}".format(
        h["epoch"], h["train_loss"], h["val_loss"], h["val_rmse"], h["val_mae"], marker))

Saved: data/models/training_history.png

Epoch      Train Loss     Val Loss   Val RMSE    Val MAE
--------------------------------------------------------
1              0.2723       0.2545     0.9344     0.7173
2              0.2602       0.2491     0.9154     0.7072
3              0.2566       0.2468     0.9061     0.7029
4              0.2544       0.2450     0.9023     0.6992
5              0.2514       0.2442     0.9025     0.6974
6              0.2489       0.2416     0.8948     0.6918
7              0.2473       0.2407     0.8936     0.6895
8              0.2457       0.2398     0.8909     0.6873
9              0.2443       0.2392     0.8895     0.6867
10             0.2432       0.2400     0.8917     0.6872
11             0.2422       0.2387     0.8884     0.6846
12             0.2413       0.2383     0.8837     0.6847 <-- best


In [7]:
# ──  Ranking Metrics — Precision@K, Recall@K, NDCG@K ──────
import numpy as np, math
from tqdm.auto import tqdm
from collections import defaultdict

model.eval()

LIKE_THRESHOLD = 4.0
K_VALUES       = [5, 10, 20]

user_test_seqs = defaultdict(list)
for seq in test_sequences:
    user_test_seqs[int(seq['user_id'])].append(seq)

print(f"Users with test sequences: {len(user_test_seqs):,}")
print(f"Evaluating at K = {K_VALUES}\n")

results_pk   = {k: [] for k in K_VALUES}
results_rk   = {k: [] for k in K_VALUES}
results_ndcg = {k: [] for k in K_VALUES}
all_movie_idxs = torch.arange(NUM_MOVIES, dtype=torch.long)
processed = 0

for user_id, seqs in tqdm(user_test_seqs.items(), desc="Ranking users"):
    gt = {movie_id_to_idx[seq['target_movie_id']]: float(seq['target_rating']) for seq in seqs}
    liked_set = {idx for idx, r in gt.items() if r >= LIKE_THRESHOLD}
    if not liked_set:
        continue

    last_seq      = seqs[-1]
    ctx_movie_t   = torch.tensor([[movie_id_to_idx[m] for m in last_seq['seq_movie_ids']]], dtype=torch.long).to(device)
    ctx_ratings_t = torch.tensor([[int(r) for r in last_seq['seq_ratings']]], dtype=torch.long).to(device)
    ctx_pos_t     = torch.tensor([last_seq['seq_positions']], dtype=torch.long).to(device)
    uf            = user_features_tensor[user_id_to_idx[user_id]].unsqueeze(0).to(device)

    all_scores = []
    with torch.no_grad():
        for start in range(0, NUM_MOVIES, 512):
            end      = min(start + 512, NUM_MOVIES)
            b        = end - start
            tgt_idxs = all_movie_idxs[start:end].to(device)
            tmf      = movie_features_tensor[tgt_idxs.cpu()].to(device)
            pred, _, _, _ = model(uf.expand(b,-1), ctx_movie_t.expand(b,-1),
                                  ctx_ratings_t.expand(b,-1), ctx_pos_t.expand(b,-1),
                                  tgt_idxs, tmf)
            all_scores.append(pred.cpu())
    all_scores = torch.cat(all_scores)

    seen = set(movie_id_to_idx[m] for m in last_seq['seq_movie_ids'])
    for idx in seen:
        all_scores[idx] = -999.0
    ranked = all_scores.argsort(descending=True).tolist()

    for k in K_VALUES:
        top_k = ranked[:k]
        hits  = sum(1 for idx in top_k if idx in liked_set)
        precision  = hits / k
        recall     = hits / len(liked_set)
        ideal_dcg  = sum(1/math.log2(i+2) for i in range(min(len(liked_set),k)))
        dcg        = sum(1/math.log2(r+2) for r,idx in enumerate(top_k) if idx in liked_set)
        ndcg       = dcg/ideal_dcg if ideal_dcg > 0 else 0.0
        results_pk[k].append(precision)
        results_rk[k].append(recall)
        results_ndcg[k].append(ndcg)
    processed += 1

print(f"\nRanking metrics over {processed:,} users\n")
print(f"{'Metric':<18}", "  ".join(f"K={k:>2}" for k in K_VALUES))
print("-" * 45)
for label, res in [("Precision@K", results_pk), ("Recall@K", results_rk), ("NDCG@K", results_ndcg)]:
    vals = [np.mean(res[k]) for k in K_VALUES]
    print(f"{label:<18}", "  ".join(f"{v:.4f}" for v in vals))

Users with test sequences: 6,040
Evaluating at K = [5, 10, 20]



Ranking users:   0%|          | 0/6040 [00:00<?, ?it/s]


Ranking metrics over 5,975 users

Metric             K= 5  K=10  K=20
---------------------------------------------
Precision@K        0.0164  0.0154  0.0141
Recall@K           0.0032  0.0062  0.0112
NDCG@K             0.0166  0.0162  0.0161


In [12]:
print(movies_enriched.columns.tolist())

['MovieID', 'Title_Original', 'Title_Clean', 'Year', 'Genres_ML', 'Genres_TMDB', 'overview', 'Keywords', 'Cast', 'Director', 'budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count', 'Genre_Vector', 'Text_For_Embedding', 'Keywords_Text', 'Overview_Embedding', 'Keyword_Embedding', 'Feature_Vector']


In [13]:
# ── Error Analysis ────────────────────────────────────────
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict

model.eval()
movie_errors = defaultdict(list)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Collecting errors"):
        user_idx         = batch['user_idx'].to(device)
        ctx_movie_idxs   = batch['ctx_movie_idxs'].to(device)
        ctx_ratings      = batch['ctx_ratings'].to(device)
        ctx_positions    = batch['ctx_positions'].to(device)
        target_movie_idx = batch['target_movie_idx'].to(device)
        target_rating    = batch['target_rating'].to(device)
        uf  = user_features_tensor[user_idx.cpu()].to(device)
        tmf = movie_features_tensor[target_movie_idx.cpu()].to(device)
        pred, _, _, _ = model(uf, ctx_movie_idxs, ctx_ratings, ctx_positions, target_movie_idx, tmf)
        errs = (pred - target_rating).abs().cpu().numpy()
        for idx, err in zip(target_movie_idx.cpu().numpy(), errs):
            movie_errors[int(idx)].append(float(err))

rows = []
for idx, errs in movie_errors.items():
    mid  = idx_to_movie_id[idx]
    row  = movies_enriched[movies_enriched['MovieID'] == mid]
    title = row['Title_Clean'].values[0] if len(row) else f"MovieID {mid}"
    genre = row['Genres_ML'].values[0] if len(row) else "Unknown"
    rows.append({'movie_idx':idx,'movie_id':mid,'title':title,'genre':genre,
                 'mean_abs_err':np.mean(errs),'n_samples':len(errs)})

error_df = pd.DataFrame(rows).sort_values('mean_abs_err', ascending=False)
filtered  = error_df[error_df['n_samples'] >= 10]

print("=" * 70)
print("TOP 15 HARDEST MOVIES TO PREDICT")
print(f"{'Title':<42} {'Genre':<18} {'MAE':>6} {'N':>6}")
print("-" * 70)
for _, r in filtered.head(15).iterrows():
    print(f"{str(r['title'])[:40]:<42} {str(r['genre'])[:16]:<18} {r['mean_abs_err']:>6.3f} {r['n_samples']:>6}")

print("\n" + "=" * 70)
print("TOP 15 EASIEST MOVIES TO PREDICT")
print(f"{'Title':<42} {'Genre':<18} {'MAE':>6} {'N':>6}")
print("-" * 70)
for _, r in filtered.tail(15).iloc[::-1].iterrows():
    print(f"{str(r['title'])[:40]:<42} {str(r['genre'])[:16]:<18} {r['mean_abs_err']:>6.3f} {r['n_samples']:>6}")

print("\n" + "=" * 50)
print("MEAN ABS ERROR BY PRIMARY GENRE")
print("=" * 50)
error_df['primary_genre'] = error_df['genre'].apply(lambda g: str(g).split('|')[0] if pd.notna(g) else 'Unknown')
genre_err = error_df[error_df['n_samples']>=5].groupby('primary_genre')['mean_abs_err'].mean().sort_values(ascending=False)
for genre, mae in genre_err.items():
    bar = "||" * int(mae * 20)
    print(f"  {genre:<20} {mae:.3f}  {bar}")

TOP 15 HARDEST MOVIES TO PREDICT
Title                                      Genre                 MAE      N
----------------------------------------------------------------------
Barenaked in America                       Documentary         1.612     14
Hana-bi                                    Comedy|Crime|Dra    1.369     10
Braindead                                  Comedy|Horror       1.307     16
Where the Buffalo Roam                     Comedy              1.277     25
Shakes the Clown                           Comedy              1.254     16
It Takes Two                               Comedy              1.220     13
Plan 9 from Outer Space                    Horror|Sci-Fi       1.209     60
8 1/2 Women                                Comedy              1.182     17
Source, The                                Documentary         1.179     11
Blood For Dracula (Andy Warhol's Dracula   Horror              1.172     11
Beyond the Mat                             Documentary      

In [14]:
# ── Movie Embedding t-SNE Visualisation ───────────────────
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from collections import Counter

ncf_emb = model.ncf_tower.target_movie_emb.weight.data.cpu().numpy()  # [N, D]
seq_emb = model.seq_tower.movie_emb.weight.data.cpu().numpy()         # [N, D]

genre_labels = []
for idx in range(NUM_MOVIES):
    mid = idx_to_movie_id[idx]
    row = movies_enriched[movies_enriched['MovieID'] == mid]
    if len(row) > 0:
        g = str(row['Genres_ML'].values[0])
        genre_labels.append(g.split('|')[0] if g != 'nan' else 'Unknown')
    else:
        genre_labels.append('Unknown')

top_genres = [g for g,_ in Counter(genre_labels).most_common(10)]
mask       = np.array([g in top_genres for g in genre_labels])
labels     = np.array(genre_labels)[mask]
cmap       = plt.cm.get_cmap('tab10', len(top_genres))

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, emb_matrix, title in zip(axes,
    [ncf_emb[mask], seq_emb[mask]],
    ["NCF Tower Embeddings", "Sequential Tower Embeddings"]):
    print(f"Running t-SNE for {title}...")
    coords = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42).fit_transform(emb_matrix)
    for i, genre in enumerate(top_genres):
        gm = (labels == genre)
        ax.scatter(coords[gm,0], coords[gm,1], c=[cmap(i)], label=genre, alpha=0.5, s=10, linewidths=0)
    ax.set_title(f"{title} (t-SNE)", fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', markerscale=2, fontsize=8)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("data/models/embedding_tsne.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: data/models/embedding_tsne.png")

/tmp/ipykernel_695/3459985034.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap       = plt.cm.get_cmap('tab10', len(top_genres))
/opt/conda/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Running t-SNE for NCF Tower Embeddings...
Running t-SNE for Sequential Tower Embeddings...


/opt/conda/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Saved: data/models/embedding_tsne.png


In [15]:
# ──Save Inference Artifacts ──────────────────────────────
import pickle
from pathlib import Path

Path("data/inference").mkdir(parents=True, exist_ok=True)

model.cpu().eval()

inference_bundle = {
    "model_state_dict"  : model.state_dict(),
    "model_config"      : {
        "num_movies"    : NUM_MOVIES,
        "movie_feat_dim": MOVIE_FEAT_DIM,
        "emb_dim"       : 128,
        "seq_len"       : SEQUENCE_LENGTH,
    },
    "movie_id_to_idx"   : movie_id_to_idx,
    "idx_to_movie_id"   : idx_to_movie_id,
    "user_id_to_idx"    : user_id_to_idx,
    "movie_features_np" : movie_features_tensor.numpy(),
    "user_features_np"  : user_features_tensor.numpy(),
    "movies_enriched"   : movies_enriched,
    "all_genres"        : all_genres,
    "training_history"  : checkpoint["history"],
    "best_val_rmse"     : checkpoint["val_rmse"],
}

with open("data/inference/inference_bundle.pkl", "wb") as f:
    pickle.dump(inference_bundle, f)

print("Saved: data/inference/inference_bundle.pkl")
print(f"  Model config  : {inference_bundle['model_config']}")
print(f"  Best val RMSE : {inference_bundle['best_val_rmse']:.4f}")

model.to(device)

Saved: data/inference/inference_bundle.pkl
  Model config  : {'num_movies': 3706, 'movie_feat_dim': 792, 'emb_dim': 128, 'seq_len': 15}
  Best val RMSE : 0.8837


HybridRecommender(
  (movie_emb): Embedding(3706, 128, padding_idx=0)
  (content_tower): ContentTower(
    (gender_emb): Embedding(3, 8)
    (age_emb): Embedding(8, 16)
    (occupation_emb): Embedding(22, 32)
    (user_mlp): Sequential(
      (0): Linear(in_features=184, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=128, bias=True)
    )
    (movie_mlp): Sequential(
      (0): LayerNorm((792,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=792, out_features=256, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=256, out_features=128, bias=True)
      (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (ncf_tower): NCFTower(
    (rating_encoder): RatingEncoder(
      (movie_emb): Embedding(3706, 128, padding_idx=0)
      (rating_emb): Embedding(6, 128, padding_idx=0)
      (attention): MultiheadAttention(
     

In [17]:
# ──  Inference Function ────────────────────────────────────
import torch, numpy as np
from typing import List, Dict, Any

def get_recommendations(
    demographics     : Dict[str, Any],
    rated_movies     : List[Dict[str, Any]],
    model, movie_id_to_idx, idx_to_movie_id,
    movies_enriched, movie_features_np, device,
    top_n=20, score_batch=512,
) -> List[Dict[str, Any]]:
    """
    demographics : {"gender": "M"/"F", "age": int (ML age group e.g. 25),
                    "occupation": int (0-20)}
    rated_movies : [{"movie_id": int, "rating": 1-5, "order": 0-14}, ...]
    Returns      : [{"movie_id", "title", "genres", "predicted_rating"}, ...]
    """
    model.eval()
    AGE_MAP = {1:0, 18:1, 25:2, 35:3, 45:4, 50:5, 56:6}
    gender_idx     = 0 if str(demographics["gender"]).upper() == "M" else 1
    age_idx        = AGE_MAP.get(int(demographics["age"]), 2)
    occupation_idx = int(demographics["occupation"])
    user_feat = torch.tensor([[gender_idx, age_idx, occupation_idx]], dtype=torch.long).to(device)

    rated_sorted = sorted(rated_movies, key=lambda x: x["order"])[:15]
    SEQ_LEN = 15
    ctx_mids, ctx_rats, ctx_pos, seen = [], [], [], set()
    for item in rated_sorted:
        mid = int(item["movie_id"])
        if mid not in movie_id_to_idx: continue
        ctx_mids.append(movie_id_to_idx[mid])
        ctx_rats.append(int(item["rating"]))
        ctx_pos.append(int(item["order"]))
        seen.add(mid)
    while len(ctx_mids) < SEQ_LEN:
        ctx_mids.append(0); ctx_rats.append(0); ctx_pos.append(len(ctx_mids)-1)

    cm_t = torch.tensor([ctx_mids], dtype=torch.long).to(device)
    cr_t = torch.tensor([ctx_rats], dtype=torch.long).to(device)
    cp_t = torch.tensor([ctx_pos],  dtype=torch.long).to(device)

    all_movie_idxs = torch.arange(NUM_MOVIES, dtype=torch.long)
    mf_t           = torch.tensor(movie_features_np, dtype=torch.float32)
    all_scores     = []

    with torch.no_grad():
        for start in range(0, NUM_MOVIES, score_batch):
            end = min(start + score_batch, NUM_MOVIES)
            b   = end - start
            ti  = all_movie_idxs[start:end].to(device)
            tmf = mf_t[start:end].to(device)
            pred, _, _, _ = model(user_feat.expand(b,-1), cm_t.expand(b,-1),
                                  cr_t.expand(b,-1), cp_t.expand(b,-1), ti, tmf)
            all_scores.append(pred.cpu())
    all_scores = torch.cat(all_scores)

    for mid in seen:
        if mid in movie_id_to_idx:
            all_scores[movie_id_to_idx[mid]] = -999.0
    top_idxs = all_scores.argsort(descending=True)[:top_n].tolist()

    recs = []
    for idx in top_idxs:
        mid  = idx_to_movie_id[idx]
        row  = movies_enriched[movies_enriched["MovieID"] == mid]
        recs.append({
            "movie_id"        : mid,
            "title"           : row["Title_Clean"].values[0]    if len(row) else f"Movie {mid}",
            "genres"          : row["Genres_ML"].values[0] if len(row) else "Unknown",
            "predicted_rating": round(float(all_scores[idx]), 3),
        })
    return recs

# ── Smoke test ────────────────────────────────────────────────────
print("Inference smoke test:")
print("=" * 70)
test_uid  = int(test_data["UserID"].iloc[0])
test_hist = user_movie_history.get(test_uid, [])[-15:]

demo_rated = [
    {"movie_id": h["movie_id"], "rating": h["rating"], "order": i}
    for i, h in enumerate(test_hist) if h["movie_id"] in movie_id_to_idx
]
recs = get_recommendations(
    demographics={"gender":"M","age":25,"occupation":4},
    rated_movies=demo_rated,
    model=model, movie_id_to_idx=movie_id_to_idx, idx_to_movie_id=idx_to_movie_id,
    movies_enriched=movies_enriched,
    movie_features_np=movie_features_tensor.cpu().numpy(),
    device=device, top_n=10
)
print(f"Top-10 recs for UserID={test_uid}:")
print(f"{'Rank':<5} {'Title':<45} {'Genres':<25} {'Pred':>6}")
print("-" * 85)
for i, rec in enumerate(recs, 1):
    print(f"{i:<5} {str(rec['title'])[:43]:<45} {str(rec['genres'])[:23]:<25} {rec['predicted_rating']:>6.3f}")
print("\nDone. get_recommendations() is ready for backend integration.")

Inference smoke test:
Top-10 recs for UserID=1:
Rank  Title                                         Genres                      Pred
-------------------------------------------------------------------------------------
1     Usual Suspects, The                           Crime|Thriller             4.675
2     Star Wars: Episode IV - A New Hope            Action|Adventure|Fantas    4.651
3     Godfather, The                                Action|Crime|Drama         4.642
4     Raiders of the Lost Ark                       Action|Adventure           4.621
5     Lawrence of Arabia                            Adventure|War              4.571
6     Shawshank Redemption, The                     Drama                      4.568
7     Dr. Strangelove or: How I Learned to Stop W   Sci-Fi|War                 4.564
8     Seven Samurai (The Magnificent Seven) (Shic   Action|Drama               4.551
9     One Flew Over the Cuckoo's Nest               Drama                      4.540
10    Godfather: